# Práctica guiada completa: Introducción al NLP

## Objetivo de la práctica
En esta práctica aprenderemos cómo un ordenador procesa el lenguaje natural.
Los modelos de Inteligencia Artificial no entienden palabras, sino números.
Por eso necesitamos transformar las frases en secuencias numéricas.

**En esta práctica aprenderás:**
- Qué es la tokenización
- Cómo funciona Tokenizer
- Cómo convertir texto en secuencias numéricas
- Cómo igualar la longitud de las frases con padding
- Cómo trabajar con datasets reales de NLP

### Parte 1. Importar librerías

**Explicación:**
Un Tokenizer es una herramienta que analiza textos, separa las palabras y asigna un número a cada una.

In [49]:
from tensorflow.keras.preprocessing.text import Tokenizer

### Parte 2. Crear nuestras primeras frases

Estas frases serán nuestro primer pequeño dataset.

In [50]:
sentences = [
    "I love my dog",
    "I love my cat"
]

### Parte 3. Crear el Tokenizer

`num_words` indica cuántas palabras diferentes guardará el modelo.

In [51]:
tokenizer = Tokenizer(num_words=100)
tokenizer.fit_on_texts(sentences)

### Parte 4. Ver el diccionario de palabras

El resultado será algo parecido a: `{'i':1,'love':2,'my':3,'dog':4,'cat':5}`

In [52]:
word_index = tokenizer.word_index
print(word_index)

{'i': 1, 'love': 2, 'my': 3, 'dog': 4, 'cat': 5}


> **Pregunta 1**
> ¿Qué número se ha asignado a la palabra 'dog'?

In [53]:
# A la palabra 'dog' se le ha asignado el número 4.

# El Tokenizer ordena las palabras por frecuencia de aparición: las que más
# se repiten reciben los números más bajos. Como 'i', 'love' y 'my' aparecen
# en ambas frases (2 veces cada una), se les asignan los números 1, 2 y 3.
# 'dog' y 'cat' solo aparecen una vez cada una, así que reciben el 4 y el 5
# respectivamente, en el orden en que el Tokenizer las encuentra.

### Parte 5. Convertir texto en números

Cada frase se convierte en una lista de números.

In [54]:
sequences = tokenizer.texts_to_sequences(sentences)
print(sequences)

[[1, 2, 3, 4], [1, 2, 3, 5]]


> **Pregunta 2**
> ¿Por qué cada frase tiene una longitud diferente?

In [55]:
# En este caso concreto ambas frases tienen la misma longitud (4 palabras cada una),
# así que sus secuencias también tienen longitud 4: [1, 2, 3, 4] y [1, 2, 3, 5].

# Cada frase puede tener un número distinto de palabras,
# y texts_to_sequences respeta eso: simplemente sustituye cada palabra por su
# número, sin añadir ni quitar nada. Si una frase tiene 3 palabras, su secuencia
# tendrá 3 números; si otra tiene 7 palabras, tendrá 7 números.

### Parte 6. Padding

El padding añade ceros para que todas las frases tengan el mismo tamaño.

In [56]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

padded = pad_sequences(sequences, maxlen=5)
print(padded)

[[0 1 2 3 4]
 [0 1 2 3 5]]


> **Pregunta 3**
> ¿Por qué aparece el número 0 en algunas posiciones?

In [57]:
# El 0 aparece porque hemos puesto maxlen=5, pero nuestras frases solo tienen
# 4 palabras.

# Para que todas las secuencias midan exactamente 5, pad_sequences
# rellena con ceros las posiciones que faltan.
# Por defecto el relleno se hace al principio (pre-padding), así que el resultado
# es algo como [0, 1, 2, 3, 4] en vez de [1, 2, 3, 4, 0].

# El 0 no corresponde a ninguna palabra del vocabulario, la red neuronal aprende
# a ignorarlo. Es simplemente una forma de decir: "aquí no hay palabra".

### Parte 7. Token OOV

El token OOV significa 'Out Of Vocabulary', es decir, palabras que el modelo no conoce.

In [58]:
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

### Parte 8. Probar con nuevas frases

In [59]:
test_data = [
    "I really love my dog",
    "My dog loves my manatee"
]

test_seq = tokenizer.texts_to_sequences(test_data)
print(test_seq)

[[2, 1, 3, 4, 5], [4, 5, 1, 4, 1]]


> **Pregunta 4**
> ¿Qué ocurre con las palabras que el modelo no conoce?

In [60]:
# Las palabras desconocidas (como 'really', 'loves' o 'manatee') se sustituyen
# por el token <OOV>, que tiene asignado el número 1.

# Sin el token OOV, esas palabras simplemente desaparecerían de la secuencia
# y perderíamos información sobre la estructura de la frase.
# Gracias a <OOV>, al menos la red sabe que ahí había una palabra, aunque no
# la conozca. Es como un comodín que dice "aquí va algo que no he visto antes".

### Parte 9. Descargar dataset de sarcasmo

In [61]:
!wget https://storage.googleapis.com/tensorflow-1-public/course3/sarcasm.json

--2026-03-04 18:00:25--  https://storage.googleapis.com/tensorflow-1-public/course3/sarcasm.json
Resolving storage.googleapis.com (storage.googleapis.com)... 64.233.180.207, 172.253.122.207, 142.251.163.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|64.233.180.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5643545 (5.4M) [application/json]
Saving to: ‘sarcasm.json.1’

sarcasm.json.1      100%[===================>]   5.38M  --.-KB/s    in 0.03s   

2026-03-04 18:00:25 (162 MB/s) - ‘sarcasm.json.1’ saved [5643545/5643545]



### Parte 10. Leer el dataset

In [62]:
import json

with open("sarcasm.json", 'r') as f:
    datastore = json.load(f)

### Parte 11. Extraer datos

In [63]:
sentences = []
labels = []
urls = []

for item in datastore:
    sentences.append(item['headline'])
    labels.append(item['is_sarcastic'])
    urls.append(item['article_link'])

> **Pregunta 5**
> ¿Cuántas frases contiene el dataset aproximadamente?

In [64]:
print(f"El dataset contiene {len(sentences)} frases")

# El dataset de sarcasmo contiene 26.709 frases (titulares de noticias).

# Cada entrada tiene un titular (headline), una etiqueta indicando si es sarcástico
# o no (is_sarcastic: 0 o 1), y el enlace al artículo original.

El dataset contiene 26709 frases


### Parte 12. Tokenizar el dataset

In [65]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)

### Parte 13. Aplicar padding

In [66]:
padded = pad_sequences(sequences, maxlen=40)

> **Pregunta 6**
> ¿Por qué necesitamos padding cuando trabajamos con redes neuronales?

In [67]:
# Porque las redes neuronales necesitan que todos los datos de entrada tengan
# exactamente las mismas dimensiones. Es como una tabla: cada fila debe tener
# el mismo número de columnas.

# Si una frase tiene 6 palabras y otra tiene 15, no podemos meterlas juntas
# en un batch de entrenamiento sin igualar su longitud primero.

# El padding resuelve esto fijando un maxlen y rellenando con ceros las frases
# más cortas (o recortando las más largas si superan ese límite).

# De esta forma, todas las secuencias tienen exactamente el mismo tamaño
# y la red puede procesarlas como una matriz uniforme.

### Parte 14. Dataset IMDB

In [68]:
import tensorflow_datasets as tfds

imdb, info = tfds.load("imdb_reviews", with_info=True, as_supervised=True)

### Parte 15. Separar datos

In [69]:
train_data, test_data = imdb['train'], imdb['test']

### Parte 16. Crear listas de frases y etiquetas

In [70]:
training_sentences = []
training_labels = []


# Extrae el texto de las reseñas (s) y sus etiquetas (l) del formato Tensor de TensorFlow
for s,l in train_data:
    training_sentences.append(str(s.numpy()))
    training_labels.append(l.numpy())

> **Pregunta 7**
> ¿Qué representan las etiquetas en el dataset IMDB?

In [71]:
# Las etiquetas representan el sentimiento de cada reseña de película:
#   - 0 = reseña negativa (al espectador no le gustó la película)
#   - 1 = reseña positiva (al espectador sí le gustó)

# Es un problema de clasificación binaria: dado el texto de una reseña,
# el modelo debe predecir si es positiva o negativa.
# El dataset IMDB contiene 25.000 reseñas de entrenamiento y 25.000 de test,
# equilibradas al 50% entre positivas y negativas.

### Parte 17. Tokenizar dataset IMDB

In [72]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(training_sentences)

training_sequences = tokenizer.texts_to_sequences(training_sentences)

### Parte 18. Padding dataset IMDB

In [73]:
training_padded = pad_sequences(training_sequences, maxlen=120)

### Parte 19. Efecto de los filtros del Tokenizer

**Explicación:**
Por defecto, el Tokenizer elimina signos de puntuación (comas, puntos, exclamaciones, etc.).
Si usamos `filters=''`, se conservan todos los caracteres y la puntuación se trata como parte de la palabra.

In [74]:
# Sin filtros: la puntuación se mantiene pegada a la palabra
tokenizer_no_filter = Tokenizer(num_words=100, filters='')

sentences_puntuacion = [
    "I love my dog",
    "I, love my cat"
]

tokenizer_no_filter.fit_on_texts(sentences_puntuacion)
word_index_no_filter = tokenizer_no_filter.word_index
print("Con filters='' (sin filtro):")
print(word_index_no_filter)

# Comparar con el Tokenizer normal (que sí filtra puntuación)
tokenizer_con_filter = Tokenizer(num_words=100)
tokenizer_con_filter.fit_on_texts(sentences_puntuacion)
print("\nCon filtros por defecto:")
print(tokenizer_con_filter.word_index)

# Observa que con filters='' la coma se queda pegada: 'i,' es una palabra
# diferente a 'i'. Con el filtro por defecto, ambas se tratan como 'i'.

Con filters='' (sin filtro):
{'love': 1, 'my': 2, 'i': 3, 'dog': 4, 'i,': 5, 'cat': 6}

Con filtros por defecto:
{'i': 1, 'love': 2, 'my': 3, 'dog': 4, 'cat': 5}


### Parte 20. Tokenizar frases con puntuación y OOV

Probamos con más frases que incluyen signos de puntuación e interrogación.

In [75]:
sentences_extended = [
    "I love my dog",
    "I, love my cat",
    "You love my dog!",
    "Do you think my dog is amazing?"
]

tokenizer2 = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer2.fit_on_texts(sentences_extended)
word_index2 = tokenizer2.word_index
print(word_index2)

sequences2 = tokenizer2.texts_to_sequences(sentences_extended)
print("\nSecuencias:")
print(sequences2)

{'<OOV>': 1, 'my': 2, 'love': 3, 'dog': 4, 'i': 5, 'you': 6, 'cat': 7, 'do': 8, 'think': 9, 'is': 10, 'amazing': 11}

Secuencias:
[[5, 3, 2, 4], [5, 3, 2, 7], [6, 3, 2, 4], [8, 6, 9, 2, 4, 10, 11]]


### Parte 21. Explorar un item del dataset de sarcasmo

Veamos cómo es un registro individual del dataset.

In [76]:
# Ver un ejemplo del dataset (el registro 20000)
datastore[20000]

{'article_link': 'https://www.theonion.com/pediatricians-announce-2011-newborns-are-ugliest-babies-1819572977',
 'headline': 'pediatricians announce 2011 newborns are ugliest babies in 30 years',
 'is_sarcastic': 1}

### Parte 22. Tamaño del vocabulario del dataset de sarcasmo

Veamos cuántas palabras únicas tiene el dataset completo.

In [77]:
# Reconstruimos el tokenizer sin límite de num_words para ver todo el vocabulario
tokenizer_sarcasm = Tokenizer(oov_token="<OOV>")
tokenizer_sarcasm.fit_on_texts(sentences)
word_index_sarcasm = tokenizer_sarcasm.word_index
print(f"Número de palabras en word index: {len(word_index_sarcasm)}")

Número de palabras en word index: 29657


### Parte 23. Padding 'post' en el dataset de sarcasmo

Usamos `padding='post'` para que los ceros se añadan al final en lugar del principio.

In [78]:
sequences_sarcasm = tokenizer_sarcasm.texts_to_sequences(sentences)
padded_sarcasm = pad_sequences(sequences_sarcasm, padding='post')

# Verificar: ver una frase original y su versión con padding
print("Frase original:")
print(sentences[2])
print("\nSecuencia con padding post:")
print(padded_sarcasm[2])

Frase original:
mom starting to fear son's web series closest thing she will have to grandchild

Secuencia con padding post:
[  145   838     2   907  1749  2093   582  4719   221   143    39    46
     2 10736     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0]


### Parte 24. Información del dataset IMDB

Veamos los metadatos del dataset IMDB: número de ejemplos, splits, etc.

In [79]:
print(info)

tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='/root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
    supervised_keys=('text', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <SplitInfo num_e

### Parte 25. Extraer datos de test del IMDB

Necesitamos también los datos de test para evaluar el modelo después de entrenarlo.

In [80]:
test_sentences = []
test_labels = []

for sentence, label in test_data:
    test_sentences.append(sentence.numpy().decode('UTF8'))
    test_labels.append(label.numpy())

print(f"Reseñas de test: {len(test_sentences)}")

Reseñas de test: 25000


### Parte 26. Preparar las etiquetas como arrays de NumPy

Las redes neuronales necesitan arrays de NumPy, no listas de Python.

In [81]:
import numpy as np

# Reextraer las frases de entrenamiento decodificando correctamente
training_sentences = []
training_labels = []

for sentence, label in train_data:
    training_sentences.append(sentence.numpy().decode('UTF8'))
    training_labels.append(label.numpy())

# Convertir a arrays de NumPy
training_labels_final = np.array(training_labels)
test_labels_final = np.array(test_labels)

print(f"Forma de training_labels_final: {training_labels_final.shape}")
print(f"Forma de test_labels_final: {test_labels_final.shape}")

Forma de training_labels_final: (25000,)
Forma de test_labels_final: (25000,)


### Parte 27. Definir hiperparámetros del modelo

Centralizamos los valores clave del modelo en variables para facilitar la experimentación.

In [82]:
vocab_size = 10000       # Número máximo de palabras en el vocabulario
max_length = 120         # Longitud máxima de cada secuencia
embedding_dim = 16       # Dimensión de los vectores de embedding
trunc_type = 'post'      # Truncar por el final si la secuencia es más larga
oov_tok = "<OOV>"        # Token para palabras desconocidas

### Parte 28. Tokenizar y aplicar padding a train y test del IMDB

Aplicamos la misma tokenización y padding tanto a los datos de entrenamiento como a los de test.

In [83]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(training_sentences)
word_index = tokenizer.word_index

# Tokenizar y padding del set de entrenamiento
sequences = tokenizer.texts_to_sequences(training_sentences)
padded = pad_sequences(sequences, maxlen=max_length, truncating=trunc_type)

# Tokenizar y padding del set de test
test_sequences = tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_sequences, maxlen=max_length, truncating=trunc_type)

print(f"Forma de padded (train): {padded.shape}")
print(f"Forma de test_padded: {test_padded.shape}")

Forma de padded (train): (25000, 120)
Forma de test_padded: (25000, 120)


### Parte 29. Construir el modelo de clasificación

**Arquitectura:**
- **Embedding**: Convierte cada índice de palabra en un vector denso de 16 dimensiones.
- **Flatten**: Aplana la matriz de embeddings en un vector 1D.
- **Dense (6, relu)**: Capa oculta con 6 neuronas.
- **Dense (1, sigmoid)**: Salida binaria (positivo/negativo).

In [84]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(6, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Parte 30. Compilar y entrenar el modelo

Usamos `binary_crossentropy` como función de pérdida porque es un problema de clasificación binaria.

In [85]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

num_epochs = 10
model.fit(
    padded,
    training_labels_final,
    epochs=num_epochs,
    verbose=1,
    validation_data=(test_padded, test_labels_final)
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.6276 - loss: 0.6091 - val_accuracy: 0.8236 - val_loss: 0.3910
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8964 - loss: 0.2613 - val_accuracy: 0.8267 - val_loss: 0.3972
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9725 - loss: 0.1095 - val_accuracy: 0.8098 - val_loss: 0.4865
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9946 - loss: 0.0336 - val_accuracy: 0.8097 - val_loss: 0.5803
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9991 - loss: 0.0082 - val_accuracy: 0.7978 - val_loss: 0.6763
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 1.0000 - loss: 0.0033 - val_accuracy: 0.8084 - val_loss: 0.7228
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9998 - loss: 0.0018 - val_accuracy: 0.7974 - val_loss: 0.7995
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9999 - loss: 0.0013 - val_accuracy: 0

### Parte 31. Extraer los pesos de la capa de Embedding

Los pesos del Embedding son los vectores que el modelo ha aprendido para cada palabra.
Podemos inspeccionarlos para entender cómo el modelo representa las palabras.

In [86]:
embedding_layer = model.layers[0]
embedding_weights = embedding_layer.get_weights()[0]

print(f"Forma de los pesos del embedding: {embedding_weights.shape}")
print(f"  → {embedding_weights.shape[0]} palabras, {embedding_weights.shape[1]} dimensiones cada una")

# Ver el vector de la primera palabra
print(f"\nVector de la palabra con índice 1: {embedding_weights[1]}")

Forma de los pesos del embedding: (10000, 16)
  → 10000 palabras, 16 dimensiones cada una

Vector de la palabra con índice 1: [ 0.07456047  0.04620363  0.01909645 -0.02985713  0.03553047 -0.05829475
  0.05370911 -0.01922385 -0.02891994 -0.02836281 -0.05667796  0.10636152
 -0.05376262 -0.05987143  0.0143099  -0.06975366]


### Parte 32. Exportar embeddings para visualización

Exportamos los vectores y las palabras a archivos TSV.
Estos archivos se pueden subir a [Embedding Projector](https://projector.tensorflow.org/)
para visualizar las relaciones entre palabras en 3D.

In [87]:
import io

# Diccionario inverso: de índice a palabra
reverse_word_index = tokenizer.index_word

# Crear los archivos TSV
out_v = io.open('vecs.tsv', 'w', encoding='utf-8')
out_m = io.open('meta.tsv', 'w', encoding='utf-8')

for word_num in range(1, vocab_size):
    word_name = reverse_word_index[word_num]
    word_embedding = embedding_weights[word_num]
    out_m.write(word_name + "\n")
    out_v.write('\t'.join([str(x) for x in word_embedding]) + "\n")

out_v.close()
out_m.close()

print("Archivos vecs.tsv y meta.tsv generados correctamente.")
print("Súbelos a https://projector.tensorflow.org/ para visualizarlos.")

Archivos vecs.tsv y meta.tsv generados correctamente.
Súbelos a https://projector.tensorflow.org/ para visualizarlos.


### Parte 33. Descargar archivos (solo en Google Colab)

Este código descargará automáticamente los archivos TSV.

In [88]:
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download('vecs.tsv')
    files.download('meta.tsv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusiones
El proceso general del NLP es:

**Texto → Tokenización → Secuencias → Padding → Modelo de IA → Predicción**

En esta práctica hemos recorrido todo el pipeline:
1. Tokenizar texto y crear un diccionario de palabras
2. Convertir frases a secuencias numéricas
3. Aplicar padding para igualar longitudes
4. Manejar palabras desconocidas con OOV
5. Trabajar con datasets reales (Sarcasmo e IMDB)
6. Construir y entrenar un modelo de clasificación de sentimientos
7. Extraer y visualizar los embeddings aprendidos

---

### Preguntas finales
1. ¿Por qué los modelos de IA no pueden trabajar directamente con texto?
2. ¿Qué función cumple el Tokenizer?
3. ¿Qué problema resuelve padding?
4. ¿Qué significa OOV?
5. ¿Por qué es importante limitar `num_words`?
6. ¿Qué hace la capa de Embedding en el modelo?
7. ¿Para qué sirven los archivos vecs.tsv y meta.tsv?

In [89]:
# ¿Por qué los modelos de IA no pueden trabajar directamente con texto?

# Porque internamente las redes neuronales solo hacen operaciones matemáticas:
# multiplicaciones de matrices, sumas, funciones de activación... Todo eso
# requiere números. Una palabra como "perro" no se puede multiplicar por un peso,
# pero el vector [0.05, -0.02, 0.17, ...] sí. Por eso hay que convertir
# el texto a representaciones numéricas antes de pasarlo al modelo.

In [90]:
# ¿Qué función cumple el Tokenizer?

# El Tokenizer hace dos cosas principales:
#  1) Construye un vocabulario: lee todas las frases, identifica las palabras
#     únicas y les asigna un índice numérico (las más frecuentes primero).
#  2) Convierte texto a secuencias: toma una frase y reemplaza cada palabra
#     por su número correspondiente del vocabulario.
# Es el puente entre el lenguaje humano y los números que necesita la red.

In [91]:
# ¿Qué problema resuelve padding?

# El padding resuelve el problema de las longitudes desiguales. Las frases
# del mundo real tienen distinto número de palabras, pero la red neuronal
# necesita inputs de tamaño fijo para poder procesarlos en lotes (batches).
# Padding iguala todas las secuencias a la misma longitud rellenando con ceros
# las que son más cortas y recortando las que son más largas.

In [92]:
# ¿Qué significa OOV?

# OOV viene de "Out Of Vocabulary" (fuera de vocabulario). Es un token especial
# que se usa para representar cualquier palabra que el Tokenizer no haya visto
# durante el entrenamiento (fit_on_texts). Sin este token, las palabras
# desconocidas simplemente se eliminarían de la secuencia y perderíamos
# información. Con <OOV>, al menos se mantiene la posición y la red sabe
# que ahí había una palabra, aunque no la reconozca.

In [93]:
# ¿Por qué es importante limitar num_words?

# Limitar num_words tiene varias ventajas:
#  - Reduce el tamaño del vocabulario, lo que hace el modelo más ligero y rápido.
#  - Descarta las palabras muy raras que aportan poco al significado general
#    y que el modelo no tendría suficientes ejemplos para aprender bien.
#  - Evita sobreajuste (overfitting) a términos infrecuentes.
#  - Ahorra memoria, ya que la capa de Embedding tendrá menos filas.

# Por ejemplo, con num_words=10000 nos quedamos con las 10.000 palabras
# más frecuentes, que suelen cubrir la gran mayoría del texto.

In [94]:
# ¿Qué hace la capa de Embedding en el modelo?

# La capa de Embedding toma cada índice de palabra (un número entero) y lo
# convierte en un vector denso de números reales (en nuestro caso, 16 dimensiones).
# Estos vectores se aprenden durante el entrenamiento: el modelo ajusta los
# valores para que palabras con significados similares tengan vectores parecidos.
# Es la primera capa del modelo y es fundamental para que la red pueda
# capturar relaciones semánticas entre palabras.

In [95]:
# ¿Para qué sirven los archivos vecs.tsv y meta.tsv?

# Estos archivos permiten visualizar los embeddings en herramientas como
# el Embedding Projector de TensorFlow (https://projector.tensorflow.org/).
#  - vecs.tsv contiene los vectores numéricos de cada palabra (una fila por palabra).
#  - meta.tsv contiene los nombres de las palabras correspondientes.
# Al subirlos al Embedding Projector, podemos ver en 3D cómo se agrupan
# las palabras con significados similares, lo que ayuda a entender
# qué ha aprendido el modelo.